# Semantic Book Recommender

Instructions: Click Run all to start the application.


## Features
- 7,000 books with semantic search
- Sentence Transformers embeddings
- Zero-shot classification (emotions, themes, genres)


## Install Dependencies


In [ ]:
#Package installation
!pip install sentence-transformers transformers torch chromadb langchain langchain-community gradio tqdm numpy pandas scikit-learn
print("All dependencies installed successfully!")

In [ ]:
#Making program imports

import requests
import zipfile
import io
import os
from pathlib import Path
import json
import numpy as np
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from typing import Dict, Any
print("All the necessary imports have been made!")

## Download Embeddings Data

Download the embeddings and metadata from Google Drive.

In [ ]:
os.makedirs('data', exist_ok=True)
os.makedirs('embeddings', exist_ok=True)

EMBEDDINGS_URL = "https://drive.google.com/uc?id=1TYJ-GBdc7_e058-s0EZGMeMbRkCwQqe6"

print("Downloading embeddings data (15MB) from google drive...")

try:
    response = requests.get(EMBEDDINGS_URL, stream=True, timeout=300)
    response.raise_for_status()

    # size for progress
    total_size = int(response.headers.get('content-length', 0))
    downloaded_size = 0

    # Download while showing level of progress
    content = io.BytesIO()
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            content.write(chunk)
            downloaded_size += len(chunk)
            if downloaded_size % (20 * 1024 * 1024) == 0:
                progress = (downloaded_size / total_size) * 100 if total_size > 0 else 0
                print(f"   Downloaded: {progress:.1f}% ({downloaded_size/(1024*1024):.1f}MB)")

    print("Download complete. Extracting...")

    # Extract the downloaded ZIP file
    content.seek(0)
    with zipfile.ZipFile(content) as zip_ref:
        zip_ref.extractall('.')

    print("Embeddings downloaded and extracteD!")

    if os.path.exists('embeddings_7k_full/books_metadata.json') and os.path.exists('embeddings_7k_full/embeddings.npy'):
        print("The required files have been verified.")
    else:
        print("Some files are missing. Please check the download.")

except Exception as e:
    print(f"Download failed: {e}")
    print("This might be due to network restrictions. Try again!")

## Set Up Core Components

Initialize the embedding store, vector database, and retriever system.

In [ ]:
PROJECT_ROOT = Path('.')
LOCAL_EMBEDDINGS_DIR = PROJECT_ROOT
BOOKS_DATA_FILE = LOCAL_EMBEDDINGS_DIR / "books_metadata.json"
EMBEDDINGS_FILE = LOCAL_EMBEDDINGS_DIR / "embeddings.npy"
COLLECTION_NAME = "book_recommendations"

# Load books data
print("Loading book metadata")
with open(BOOKS_DATA_FILE, 'r', encoding='utf-8') as f:
    books_data = json.load(f)
print(f"Loaded {len(books_data)} books")

# Load embeddings
print("Loading embeddings")
embeddings_array = np.load(EMBEDDINGS_FILE)
print(f"Loaded embeddings: {embeddings_array.shape}")

# Sentence Transformers
print(" Loading Sentence Transformers model")
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("Model loaded")

print("Setting up ChromaDB...")
chroma_client = chromadb.PersistentClient(path="./chroma_db")
try:
    collection = chroma_client.get_collection(name=COLLECTION_NAME)
    print("Using existing ChromaDB collection")
except:
    collection = chroma_client.create_collection(name=COLLECTION_NAME)

    print("Adding books to vector database")
    for i, book in enumerate(books_data):
        collection.add(
            ids=[f"book_{i}"],
            embeddings=[embeddings_array[i].tolist()],
            metadatas=[book]
        )

        if (i + 1) % 1000 == 0:
            print(f"Added {i + 1}/{len(books_data)} books")

    print("Vector database created")

print("Ready for semantic search!")

## Classification & Analysis

Initialize the emotion, theme, and genre classification system.

In [ ]:
EMOTION_LABELS = ["hopeful", "sad", "dark", "comforting", "uplifting", "mysterious",
    "suspenseful", "romantic", "inspiring", "melancholic", "emotional",
    "reflective", "introspective", "optimistic", "haunting"]

THEME_LABELS = [ "identity", "justice", "trauma", "colonialism", "migration",
    "self-discovery", "healing", "belonging", "resistance",
    "cultural conflict", "moral dilemmas", "existential themes",
    "personal transformation", "human nature"]

GENRE_LABELS = ["fiction", "non-fiction", "romance", "mystery", "thriller",
    "science fiction", "fantasy", "historical fiction", "biography",
    "memoir", "literary fiction", "horror", "young adult",
    "self-help", "philosophy", "political", "crime", "adventure",
    "drama", "comedy", "contemporary"]

ATMOSPHERE_LABELS = ["mysterious", "suspenseful", "eerie", "tense", "foreboding",
    "adventurous", "epic journey", "quest-driven", "introspective",
    "character-driven", "emotionally immersive", "fast-paced thriller",
    "slow reflective narrative", "lyrical", "dreamy", "gritty",
    "surreal", "claustrophobic", "intimate", "meditative"]

# zero-shot classifier
try:
    classifier = pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli",
        device=-1
    )
    print("Zero-shot classifier is ready")
except Exception as e:
    print(f"Classifier setup failed: {e}")
    classifier = None

# sentiment_analyzer = None
sentiment_analyzer = None
print("Sentiment analyzer has been disabled for faster performance")

def analyze_text(text: str) -> Dict[str, Any]:
    if not text or len(text.strip()) < 10:
        return {"emotions": [], "themes": [], "genres": [], "atmosphere": [], "sentiment": {"neutral": 0.5}}

    result = {}

    if classifier:
        try:
            emotion_result = classifier(text[:512], EMOTION_LABELS, multi_label=True)
            result["emotions"] = [
                {"label": label, "score": score}
                for label, score in zip(emotion_result["labels"][:3], emotion_result["scores"][:3])
                if score > 0.3
            ]

            theme_result = classifier(text[:512], THEME_LABELS, multi_label=True)
            result["themes"] = [
                {"label": label, "score": score}
                for label, score in zip(theme_result["labels"][:2], theme_result["scores"][:2])
                if score > 0.4
            ]

            genre_result = classifier(text[:512], GENRE_LABELS, multi_label=True)
            result["genres"] = [
                {"label": label, "score": score}
                for label, score in zip(genre_result["labels"][:2], genre_result["scores"][:2])
                if score > 0.5
            ]

            atmosphere_result = classifier(text[:512], ATMOSPHERE_LABELS, multi_label=True)
            result["atmosphere"] = [
                {"label": label, "score": score}
                for label, score in zip(atmosphere_result["labels"][:2], atmosphere_result["scores"][:2])
                if score > 0.3
            ]

        except Exception as e:
            print(f"Classification error: {e}")
            result.update({"emotions": [], "themes": [], "genres": [], "atmosphere": [] })

    # Sentiment analysis
    if sentiment_analyzer:
        try:
            sentiment_result = sentiment_analyzer(text[:512])
            # Convert to simple format
            sentiment_scores = {}
            # Handle different result formats
            if isinstance(sentiment_result, list):
                for item in sentiment_result:
                    if isinstance(item, dict) and "label" in item and "score" in item:
                        label = item["label"].lower()
                        if "positive" in label or "pos" in label:
                            sentiment_scores["positive"] = item["score"]
                        elif "negative" in label or "neg" in label:
                            sentiment_scores["negative"] = item["score"]
                        elif "neutral" in label:
                            sentiment_scores["neutral"] = item["score"]

            # Ensure we have all sentiment categories
            if not sentiment_scores:
                sentiment_scores = {"neutral": 0.5}

            result["sentiment"] = sentiment_scores
        except Exception as e:
            # skip sentiment analysis errors to avoid spam
            result["sentiment"] = {"neutral": 0.5}

    return result

print("Analysis system ready!")

## Search Function



In [ ]:
def search_books(
    query: str,
    emotion_filter: str = "None",
    theme_filter: str = "None",
    genre_filter: str = "None",
    atmosphere_filter: str = "None",
    mood_filter: str = "None",
    sentiment_filter: str = "None",
    max_results: int = 8
) -> str:

    if (
        not query.strip()
        and not genre_filter
        and not emotion_filter
        and not theme_filter
        and not atmosphere_filter
        and not mood_filter
        and not sentiment_filter
    ):
        return "Please enter a query or select at least one filter."

    try:
        # Build enhanced query
        query_parts = [query.strip()]

        if emotion_filter and emotion_filter != "None":
            query_parts.append(f"{emotion_filter} emotion")
        if theme_filter and theme_filter != "None":
            query_parts.append(f"{theme_filter} theme")
        if genre_filter and genre_filter != "None":
            query_parts.append(f"{genre_filter} genre")
        if atmosphere_filter and atmosphere_filter != "None":
            query_parts.append(f"{atmosphere_filter} atmosphere")
        if mood_filter and mood_filter != "None":
            query_parts.append(f"{mood_filter} mood")
        if sentiment_filter and sentiment_filter != "None":
            query_parts.append(f"{sentiment_filter} sentiment")

        enhanced_query = " ".join(query_parts)
        print(f"Searching for: '{enhanced_query}'")

        # Generate query embedding
        if query.strip():
            query_embedding = embed_model.encode(query).tolist()

            results = collection.query(
                query_embeddings=[query_embedding],
                n_results=50
            )

            candidate_ids = results["ids"][0] if results["ids"] else []

                # If there is no query start with all the books
        else:
            # get all books
            all_books = collection.get()
            candidate_ids = all_books["ids"] or []
            results = {
                "ids": [all_books.get("ids", [])],
                "distances": [ [0] * len(candidate_ids) ],
                "metadatas": [ all_books.get("metadatas", [{}]*len(candidate_ids)) ]
            }

        if not candidate_ids:
            return f"No books found matching your criteria. Try different keywords or remove some filters."

        # Process results
        processed_results = []
        for i, (doc_id, distance, metadata) in enumerate(zip(
            results["ids"][0],
            results["distances"][0],
            results["metadatas"][0]
        )):
            # Calculate relevance score
            relevance_score = max(0, 1 - (distance / 2))
            relevance_pct = int(relevance_score * 100)

            # book info
            title = metadata.get("title", "Unknown Title")
            author = metadata.get("author", "Unknown Author")
            genre = metadata.get("genre", "Fiction")
            mood = metadata.get("mood", "General")
            description = metadata.get("description", "No description available.")

            # relevance bar
            filled = relevance_pct // 10
            relevance_bar = "█" * filled + "░" * (10 - filled)

            result = f"📚 **{title}**\n"
            result += f"👤 {author} | 🎭 {genre}\n"
            result += f"📖 {description}\n"
            result += f"🎯 {relevance_pct}% {relevance_bar}\n"

            processed_results.append((relevance_score, result))

        # Sort by relevance
        processed_results.sort(key=lambda x: x[0], reverse=True)
        top_results = [result for _, result in processed_results[:max_results]]

        # Summary
        summary = f"Found {len(top_results)} relevant books for '{query}'"
        active_filters = []
        if emotion_filter and emotion_filter != "None": active_filters.append(f"emotion: {emotion_filter}")
        if theme_filter and theme_filter != "None": active_filters.append(f"theme: {theme_filter}")
        if genre_filter and genre_filter != "None": active_filters.append(f"genre: {genre_filter}")
        if atmosphere_filter and atmosphere_filter != "None": active_filters.append(f"atmosphere: {atmosphere_filter}")
        if mood_filter and mood_filter != "None": active_filters.append(f"mood: {mood_filter}")
        if sentiment_filter and sentiment_filter != "None": active_filters.append(f"sentiment: {sentiment_filter}")

        if active_filters:
            summary += f" (filtered by: {', '.join(active_filters)})"

        return f"{summary}\n\n" + "\n\n".join(top_results)

    except Exception as e:
        return f"Search error: {str(e)}. Please try again."

print("Search function ready!")


## Create Gradio Interface

Launch the interactive web interface for book recommendations.

In [ ]:
import gradio as gr

with gr.Blocks(title="Semantic Book Recommender", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 📚 Semantic Book Recommender

    Come discover your next read! Enter a description of what you're looking for,
    and our system will find books that match your interests.

    **Features:**
    - 7,000+ books in our database
    - Sentence Transformers for semantic understanding
    - Zero-shot classification (emotions, themes, genres)
    - Sentiment analysis
    - Advanced filtering options """)

    with gr.Row():
        with gr.Column(scale=2):
            query_input = gr.Textbox(
                label="📝 What are you looking for?",
                placeholder="e.g., 'mysterious adventure with dragons and magic', 'heartwarming story about friendship', 'dark psychological thriller'",
                lines=3,
                info="Describe the type of book you're interested in using natural language"
            )

            max_results = gr.Slider(
                minimum=1,
                maximum=20,
                value=8,
                step=1,
                label="📊 Number of results"
            )

        with gr.Column(scale=1):
            gr.Markdown("""
            ### Tips
            - Be specific about themes, emotions, or genres
            - Try: "uplifting story about overcoming adversity"
            - Or: "suspenseful mystery with unexpected twists"
            - Use filters to narrow down results
            """)

    # Advanced filters in tabs
    with gr.Tabs():
        with gr.TabItem("🎭 Genre Filter"):
            genre_filter = gr.Dropdown(
                choices=["None"] + GENRE_LABELS,
                value="None",
                label="Select Genre (Optional)"
            )

        with gr.TabItem("😊 Emotion Filter"):
            emotion_filter = gr.Dropdown(
                choices=["None"] + EMOTION_LABELS,
                value="None",
                label="Select Emotional Tone (Optional)"
            )

        with gr.TabItem("🧠 Theme Filter"):
            theme_filter = gr.Dropdown(
                choices=["None"] + THEME_LABELS,
                value="None",
                label="Select Thematic Focus (Optional)"
            )

        with gr.TabItem("🌟 Atmosphere Filter"):
            atmosphere_filter = gr.Dropdown(
                choices=["None"] + ATMOSPHERE_LABELS,
                value="None",
                label="Select Reading Atmosphere (Optional)"
            )

        with gr.TabItem("🎵 Mood Filter"):
            mood_filter = gr.Dropdown(
                choices=["None", "Adventurous", "Dark", "Emotional", "Hopeful", "Whimsical", "Reflective", "Suspenseful", "Intellectual", "Inspiring"],
                value="None",
                label="Select Mood (Optional)"
            )

        with gr.TabItem("💭 Sentiment Filter"):
            sentiment_filter = gr.Dropdown(
                choices=["None", "Positive", "Negative", "Neutral"],
                value="None",
                label="Select Sentiment Preference (Optional)"
            )

    # Search button function
    search_btn = gr.Button("🔍 Search Books", variant="primary", size="lg")

    # Results output
    results_output = gr.Textbox(
        label="📚 Recommended Books",
        lines= 25,
        show_copy_button =True,
        placeholder="Your book recommendations will appear here..."
    )

    # Connect the search function
    search_btn.click(
        fn=search_books,
        inputs=[
            query_input,
            emotion_filter,
            theme_filter,
            genre_filter,
            atmosphere_filter,
            mood_filter,
            sentiment_filter,
            max_results
        ],
        outputs=results_output
    )

    gr.Examples(
        examples=[
            ["fantasy adventure with magic and dragons"],
            ["heartwarming story about friendship and love"],
            ["dark mystery with suspense and intrigue"],
            ["inspiring tale of overcoming adversity"],
            ["philosophical novel about the meaning of life"],
            ["romantic comedy with humor and heart"],
        ],
        inputs=query_input,
        label="💡 Try These Example Queries"
    )

# Launch the interface
print("\n" + "="*60)
print(" BOOK RECOMMENDER READY!")
print("="*60)
print("\n⏳ Launching interface...")
demo.launch(share=True, debug=False)

print("\n🎉 SUCCESS!")
print("Your Book Recommender is now running!")
print("Look for the PUBLIC URL above (https://xxxxx.gradio.live) and click that URL to open the interface")
print("\n💡 Keep this Colab tab open while using the interface")